# headswap — expression chain, timed

**T4 head swap → LivePortrait expression transfer.** Run order: Cell 1 (setup) → Cell 2 (upload) → Cell 3 (run).

Settings are fixed to the arm that worked: LivePortrait runs **after** the swap (before it, T4 regenerates the head and the expression is lost), `driving_multiplier=0.8`, `animation_region=lip`, and `face_refine` skipped on bust shots.

Cell 3 runs the chain **twice** — a COLD pass that pays every one-time cost, and a WARM pass which is what a served request actually costs. Read the WARM row.

All logic lives in `scripts/profile_chain.py`, pulled fresh by Cell 3. Nothing here is a `#@param` that can go stale in the browser tab — see `docs/PIPELINE_STATE.md` → "Colab workflow" for why that matters.


In [ ]:
#@title Cell 1 - Setup (run once per runtime)
from pathlib import Path
import subprocess, shutil, os, signal, sys

assert Path("/content").exists(), "Open this notebook in Google Colab."
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> GPU, then re-run.")
print(f"GPU {torch.cuda.get_device_name(0)}")

REPO = Path("/content/headswap_V2")
BRANCH = "simple-full-body-head-swap"
if not REPO.exists():
    subprocess.run(["git", "clone",
                    "https://github.com/malihashar/headswap_V2.git", str(REPO)],
                   check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH,
                f"origin/{BRANCH}"], check=True)
subprocess.run(["git", "-C", str(REPO), "reset", "--hard", f"origin/{BRANCH}"],
               check=True)
print("HEAD:", subprocess.getoutput(f"git -C {REPO} rev-parse --short HEAD"))

os.chdir(REPO)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)

FRESH = not Path("/content/ComfyUI/server.py").exists()
if FRESH:
    print("-> Fresh runtime: installing ComfyUI + Krea2 weights (~20GB, slow)")
    shutil.rmtree("/content/ComfyUI", ignore_errors=True)
    r = subprocess.run(["bash", "scripts/setup_colab.sh", "--no-drive", "--krea2"],
                       check=False, capture_output=True, text=True)
    print("setup exit:", r.returncode)
    print(r.stdout[-1200:])
    if r.returncode != 0:
        print(r.stderr[-2000:]); raise SystemExit("setup_colab.sh failed")
    subprocess.run(["pip", "install", "-q", "--no-cache-dir", "--force-reinstall",
                    "--no-deps", "numpy==2.4.6"], check=True)
    print("\n✓ Restarting kernel (expected). Then run Cell 2.")
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print("ComfyUI already present - skipping the slow install.")
    print("✓ Ready. Run Cell 2.  (LivePortrait auto-installs in Cell 3.)")


In [ ]:
#@title Cell 2 - Upload a pair
# BODY  = the photo you keep (pose, clothing, background) AND whose
#         expression you want on the final face.
# FACE  = the donor whose identity is transferred in.
import os, uuid
from pathlib import Path
from PIL import Image
from IPython.display import display
from google.colab import files

REPO = Path("/content/headswap_V2")
PAIR = REPO / "data" / "custom" / "chain_pair"
PAIR.mkdir(parents=True, exist_ok=True)
for old in PAIR.glob("*"):
    old.unlink()

def grab(role, what):
    print(f"\n=== Upload the {role.upper()} image ({what}) ===")
    up = files.upload()
    if not up:
        raise SystemExit(f"No {role} uploaded - re-run this cell.")
    name = next(iter(up))
    Image.open(name).convert("RGB").save(PAIR / f"{role}.png")
    os.remove(name)
    im = Image.open(PAIR / f"{role}.png")
    print(f"  saved {role}: {im.size[0]}x{im.size[1]}")
    return im

body = grab("body", "TARGET: pose/clothes/background + the expression you want")
face = grab("face", "DONOR: the identity to transfer in")
display(body); display(face)
print("\n✓ Ready. Run Cell 3.")


In [ ]:
#@title Cell 3 - Run the chain and time it
import subprocess
from pathlib import Path
from IPython.display import Image as IPImage, display, Markdown

REPO = Path("/content/headswap_V2")
subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=False)
print("HEAD:", subprocess.getoutput(f"git -C {REPO} rev-parse --short HEAD"))

OUT = REPO / "results" / "chain_run"
LOG = Path("/content/chain_run.log")

# Fixed to the arm that worked. LivePortrait runs AFTER the swap: run before
# it and T4 regenerates the head, which normalises the edited expression away
# (measured -- an opened mouth came back as an ordinary closed smile).
cmd = [
    "python", "-u", "scripts/profile_chain.py",
    "--pair-dir", "data/custom/chain_pair",
    "--out-dir", str(OUT),
    "--repeats", "1",
    "--skip-refine",
    "--lp-order", "after",
    "--driving-multiplier", "0.8",
    "--animation-region", "lip",
]
print("running (first pass is COLD and slow; read the WARM row) ...\n")
with open(LOG, "w") as fh:
    subprocess.run(cmd, cwd=str(REPO), stdout=fh, stderr=subprocess.STDOUT,
                   check=False)

summary = OUT / "SUMMARY.txt"
if summary.exists():
    print(summary.read_text())
else:
    print("No summary - last 40 log lines:")
    print("\n".join(LOG.read_text().splitlines()[-40:]))

display(Markdown("### T4 swap only"))
p = OUT / "swap_only_warm1.png"
if p.exists():
    display(IPImage(filename=str(p)))
display(Markdown("### Final (after LivePortrait expression transfer)"))
p = OUT / "final_warm1.png"
if p.exists():
    display(IPImage(filename=str(p)))

print(f"\nfull log: {LOG}")
